# 02 — Pruning Analysis

**Goal:** compare full tokens vs. random pruning vs. cross-attention (text-conditioned) pruning at 25% / 50% / 75% token removal on a handful of images, and run the D3 (score collapse) + D4 (spatial preservation) diagnostics.

**Expected:** random pruning is a *surprisingly strong* baseline; the question is whether text-conditioned scoring matches or beats it at matched token budgets (D7).

In [ ]:
!pip -q install torch transformers accelerate Pillow requests
import sys, os, torch, requests
sys.path.insert(0, os.path.join(os.getcwd(), "src"))
from PIL import Image
from io import BytesIO
from transformers import AutoProcessor, AutoModelForImageTextToText
from tgtp.models.smolvlm_wrapper import SmolVLMPruningWrapper
from tgtp.evaluation.diagnostics import score_distribution_stats, spatial_preservation, random_vs_smart_delta

MODEL = "HuggingFaceTB/SmolVLM-Instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32
processor = AutoProcessor.from_pretrained(MODEL, trust_remote_code=True)
model = AutoModelForImageTextToText.from_pretrained(MODEL, torch_dtype=dtype, trust_remote_code=True).to(device)
model.eval()
wrapper = SmolVLMPruningWrapper(model)

In [ ]:
# Small eval set: 2 images x 3 questions with a tiny keyword-F1 scorer (Colab-light).
urls = [
    "https://images.unsplash.com/photo-1502877338535-766e1452684a?w=600",
    "https://images.unsplash.com/photo-1500530855697-b586d89ba3ee?w=600",
]
images = [Image.open(BytesIO(requests.get(u, timeout=30).content)).convert("RGB").resize((384, 384)) for u in urls]
questions = ["What color is the car?", "What is the weather?", "Are there people on the street?"]
gold = ["turquoise", "sunny", "no"]  # rough expected answers for plotting only

def answer(keep_ratio, scoring):
    outs = []
    for img, q in zip(images, questions):
        pv = processor(images=img, return_tensors="pt")["pixel_values"].to(device)
        enc = processor(text=q, return_tensors="pt")
        gen, _, _ = wrapper.generate_with_pruning(
            pv, enc["input_ids"].to(device), enc["attention_mask"].to(device),
            keep_ratio=keep_ratio, scoring_method=scoring, max_new_tokens=12)
        outs.append(processor.decode(gen[0, enc["input_ids"].shape[1]:], skip_special_tokens=True))
    return outs

In [ ]:
import pandas as pd
rows = []
for ratio in [0.0, 0.25, 0.5, 0.75]:
    if ratio == 0.0:
        ans = []
        for img, q in zip(images, questions):
            pv = processor(images=img, return_tensors="pt")["pixel_values"].to(device)
            enc = processor(text=q, return_tensors="pt")
            out = model.generate(**{**{"pixel_values": pv}, **{k: enc[k].to(device) for k in ["input_ids", "attention_mask"]}}, max_new_tokens=12)
            ans.append(processor.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True))
        rows.append(("full", ratio, ans))
    else:
        rows.append(("random", ratio, answer(1 - ratio, "random")))
        rows.append(("smart", ratio, answer(1 - ratio, "cross_attention")))

pd.DataFrame([(m, r, [a[:30] for a in ans]) for m, r, ans in rows], columns=["method", "ratio", "answers"])

In [ ]:
# Diagnostics: D3 (score collapse) + D4 (spatial preservation) at 50% pruning.
pv = processor(images=images[0], return_tensors="pt")["pixel_values"].to(device)
enc = processor(text=questions[0], return_tensors="pt")
with torch.no_grad():
    visual = wrapper.extract_visual_tokens(pv)
    scores = wrapper.score(visual, enc["input_ids"].to(device), enc["attention_mask"].to(device), scoring_method="cross_attention")
    _, keep_idx = wrapper.select_tokens(visual, scores, 0.5)

d3 = score_distribution_stats(scores[0])
d4 = spatial_preservation(keep_idx[0].tolist(), grid_h=9, grid_w=9)
print("D3 score stats:", d3)
print("D4 spatial preservation:", d4)
print("D7-style delta (smart - random) at 50%:", random_vs_smart_delta({0.5: 0.0}, {0.5: 0.0}))

**Interpretation:**
- D3: if `std` ≈ 0 the scores have collapsed — the scorer provides no signal (fix: entropy loss in training).
- D4: after 50% pruning, row/column coverage should stay high for spatial tasks (GQA).
- D7: for each ratio, smart must beat random at a *matched token budget* for the thesis claim to hold.